# 🔴 Pokémon vs. 🟦 Digimon: *Why does Machine Learning actually work?*

### An interactive demo of the **generalization bound** and the **model-complexity trade-off**

> Based on Hung-yi Lee's *"淺談機器學習原理"* (Basic Principles of ML).

We train a classifier on a **small handful** of examples, yet we trust it on creatures
it has **never seen**. Why is that ever allowed? This notebook turns that abstract
probability question into a simulation you can poke at.

The whole story rests on one inequality:

$$P(\mathcal{D}_{train}\ \text{is bad}) \;\le\; |\mathcal{H}| \cdot 2\exp\!\left(-2N\varepsilon^2\right)$$

By the end you will *see* why **more data (larger $N$)** and a **simpler model
(smaller $|\mathcal{H}|$)** are what make learning trustworthy — and why we can't just
make $|\mathcal{H}|$ tiny and call it a day.

## 🎯 Learning objectives

After running this notebook you should be able to:

1. State the 3-step recipe of ML: **(1)** model with unknown parameters → **(2)** loss → **(3)** optimization.
2. Distinguish **training loss** $L(h_{train}, \mathcal{D}_{train})$ from **true loss** $L(h_{train}, \mathcal{D}_{all})$, and explain the **generalization gap**.
3. Explain why a *bad* training set is one where *some* hypothesis looks very different on $\mathcal{D}_{train}$ vs. $\mathcal{D}_{all}$.
4. Use **Hoeffding's inequality + the union bound** to bound the probability of drawing a bad training set.
5. Predict how that bound changes with the **dataset size $N$** and **model complexity $|\mathcal{H}|$**, and verify it against Monte-Carlo simulation.
6. Describe the **bias–complexity trade-off**: why both *too-simple* and *too-complex* models hurt true loss.

## 0. Setup

Lightweight stack only: `numpy`, `matplotlib`, `ipywidgets` — all pre-installed in
Google Colab. No GPU needed; everything here is a fast CPU simulation.

In [ ]:
# === Setup: imports + reproducible randomness ===========================================
import sys, subprocess

# ipywidgets ships with Colab; install only if somehow missing (keeps the notebook portable).
try:
    import ipywidgets  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"], check=False)

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, FloatLogSlider

# Makes ipywidgets sliders render reliably inside Google Colab (harmless elsewhere).
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

SEED = 42                      # change this to spawn a different Pokémon/Digimon universe
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
print("Setup complete  |  numpy", np.__version__)

## 1. The concept in one screen

We want a function $f$ that takes a creature image and outputs **Pokémon** or **Digimon**.

**Domain knowledge (slide observation):** Digimon are drawn with *more complex lines* than
Pokémon. So we summarise each image with a single number:

$$e = \text{"edge complexity"} \quad(\text{how tangled the outline is}).$$

**Step 1 — model with an unknown parameter.** A threshold classifier:

$$f_h(e) = \begin{cases}\text{Digimon (1)} & e \ge h\\[2pt] \text{Pokémon (0)} & e < h\end{cases}$$

The unknown parameter is the threshold $h$. The set of all candidate thresholds is the
**hypothesis set** $\mathcal{H} = \{1, 2, \dots, 10000\}$, and $|\mathcal{H}|$ is the
**model complexity**.

**Step 2 — loss.** The error rate (0/1 loss), which always lives in $[0,1]$:

$$L(h, \mathcal{D}) = \frac{1}{N}\sum_{n=1}^{N} \mathbb{1}\big[f_h(x_n)\neq \hat y_n\big].$$

**Step 3 — optimization.** Pick the threshold with the smallest loss on the data we have.

The catch: we optimise on a small **training set** $\mathcal{D}_{train}$, but we *care*
about the loss over **everything**, $\mathcal{D}_{all}$.

## 2. Build the universe $\mathcal{D}_{all}$

In real life you can never collect *all* Pokémon and Digimon. Here we *can*: we simulate
the entire universe so we know the ground-truth answers and can check whether learning
from a small sample actually works.

Pokémon (simpler lines → smaller $e$) and Digimon (complex lines → larger $e$) **overlap**,
so no single threshold is ever perfect — exactly like the slides where the best possible
error is about $0.28$.

In [ ]:
# === Build D_all: the full population we will sample training sets from ==================
N_ALL        = 20000          # size of the "whole universe"
E_MIN, E_MAX = 1, 10000       # edge-complexity range  ->  H = {1, ..., 10000}

def make_universe(n=N_ALL, seed=SEED):
    g = np.random.default_rng(seed)
    n_poke = n // 2
    n_digi = n - n_poke
    # Two overlapping Gaussians in edge-complexity, clipped to the valid range.
    e_poke = g.normal(3800, 1600, n_poke)   # Pokémon: simpler  -> lower  edge count
    e_digi = g.normal(5800, 1600, n_digi)   # Digimon: complex  -> higher edge count
    e = np.clip(np.concatenate([e_poke, e_digi]), E_MIN, E_MAX)
    y = np.concatenate([np.zeros(n_poke), np.ones(n_digi)]).astype(int)  # 0=Pokémon, 1=Digimon
    order = g.permutation(n)
    return e[order], y[order]

e_all, y_all = make_universe()
print(f"D_all: {len(e_all)} creatures   Pokémon={np.sum(y_all==0)}   Digimon={np.sum(y_all==1)}")

# Look at the two overlapping populations.
plt.figure()
plt.hist(e_all[y_all == 0], bins=40, alpha=0.6, label="Pokémon (0)")
plt.hist(e_all[y_all == 1], bins=40, alpha=0.6, label="Digimon (1)")
plt.xlabel("e = edge complexity"); plt.ylabel("count")
plt.title("D_all: the classes overlap, so perfect accuracy is impossible")
plt.legend(); plt.show()

## 3. Find the best-in-the-universe threshold $h_{all}$

If we *could* see all of $\mathcal{D}_{all}$, we would just pick

$$h_{all} = \arg\min_{h\in\mathcal{H}} L(h, \mathcal{D}_{all}).$$

This is the **gold standard** — the best any threshold classifier can possibly do. We will
keep comparing our small-sample answers against it.

In [ ]:
# === Threshold classifier, its loss, and the optimizer ==================================
def predict_threshold(e, h):
    """f_h(e): predict Digimon (1) if e >= h, else Pokémon (0)."""
    return (e >= h).astype(int)

def loss_threshold(h, e, y):
    """L(h, D) = error rate for a single threshold h (clear but slow)."""
    return np.mean(predict_threshold(e, h) != y)

def threshold_loss_curve(hs, e, y):
    """Vectorized L(h, D) for MANY thresholds at once (fast: uses sorted searchsorted).

    error = (#Pokémon with e>=h, wrongly called Digimon) + (#Digimon with e<h, wrongly called Pokémon)
    """
    pe = np.sort(e[y == 0]); de = np.sort(e[y == 1])
    poke_ge = len(pe) - np.searchsorted(pe, hs, side="left")  # Pokémon predicted Digimon
    digi_lt = np.searchsorted(de, hs, side="left")            # Digimon predicted Pokémon
    return (poke_ge + digi_lt) / len(e)

H = np.arange(E_MIN, E_MAX + 1)               # the full hypothesis set H = {1, ..., 10000}
loss_all_curve = threshold_loss_curve(H, e_all, y_all)
h_all = int(H[np.argmin(loss_all_curve)])
L_h_all = float(loss_all_curve.min())
print(f"Best threshold on the WHOLE universe:  h_all = {h_all}")
print(f"Its (unbeatable) loss  L(h_all, D_all) = {L_h_all:.3f}")

plt.figure()
plt.plot(H, loss_all_curve, lw=2)
plt.axvline(h_all, color="red", ls="--", label=f"h_all = {h_all}")
plt.scatter([h_all], [L_h_all], color="red", zorder=5)
plt.xlabel("threshold h"); plt.ylabel("L(h, D_all) = error rate")
plt.title("Loss over the whole universe for every candidate threshold")
plt.legend(); plt.show()

## 4. Reality: we only get a small sample $\mathcal{D}_{train}$

In practice we draw a small training set i.i.d. from the universe and optimise on *that*:

$$h_{train} = \arg\min_{h\in\mathcal{H}} L(h, \mathcal{D}_{train}).$$

Two losses now matter and they are **not** the same:

| quantity | meaning |
|---|---|
| $L(h_{train}, \mathcal{D}_{train})$ | **training loss** — what we measure, looks optimistic |
| $L(h_{train}, \mathcal{D}_{all})$ | **true loss** — what we actually get in the wild |

Their difference vs. the gold standard, $L(h_{train},\mathcal{D}_{all}) - L(h_{all},\mathcal{D}_{all})$,
is the **generalization gap**. Move the sliders below: each new sample gives a *different*
$h_{train}$ and a *different* gap — sometimes the training loss even dips **below** the
gold standard, which is exactly the trap (slides p.15–16).

In [ ]:
# === Sample a training set, fit it, and compare training-loss vs. true-loss =============
def sample_train(n, seed=None):
    g = np.random.default_rng(seed)
    idx = g.choice(len(e_all), size=n, replace=False)   # i.i.d. draw from D_all
    return e_all[idx], y_all[idx]

def fit_threshold(e, y):
    """Return (h_train, training_loss) = argmin over H of L(h, D_train)."""
    curve = threshold_loss_curve(H, e, y)
    return int(H[np.argmin(curve)]), float(curve.min())

def explore_one_sample(N=200, sample_id=1):
    e_tr, y_tr = sample_train(N, seed=sample_id)
    h_tr, L_train = fit_threshold(e_tr, y_tr)
    L_true = float(threshold_loss_curve(np.array([h_tr]), e_all, y_all)[0])

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ax[0].hist(e_tr[y_tr == 0], bins=20, alpha=0.6, label="Pokémon")
    ax[0].hist(e_tr[y_tr == 1], bins=20, alpha=0.6, label="Digimon")
    ax[0].axvline(h_tr,  color="red",   ls="--", label=f"h_train = {h_tr}")
    ax[0].axvline(h_all, color="green", ls=":",  label=f"h_all = {h_all}")
    ax[0].set_title(f"This particular D_train (N={N}, sample #{sample_id})")
    ax[0].set_xlabel("e"); ax[0].legend()

    ax[1].plot(H, loss_all_curve, label="L(h, D_all)  (true)")
    ax[1].plot(H, threshold_loss_curve(H, e_tr, y_tr), label="L(h, D_train)  (sample)")
    ax[1].axvline(h_tr, color="red", ls="--"); ax[1].axvline(h_all, color="green", ls=":")
    ax[1].set_xlabel("threshold h"); ax[1].set_ylabel("loss")
    ax[1].set_title("Training curve wiggles around the true curve"); ax[1].legend()
    plt.tight_layout(); plt.show()

    gap = L_true - L_h_all
    print(f"L(h_train, D_train) = {L_train:.3f}   <- training loss (optimistic)")
    print(f"L(h_train, D_all)   = {L_true:.3f}   <- TRUE loss (what we really get)")
    print(f"L(h_all,  D_all)    = {L_h_all:.3f}   <- gold standard")
    print(f"generalization gap  = {gap:+.3f}")

interact(explore_one_sample,
         N=IntSlider(value=200, min=20, max=2000, step=20, description="N"),
         sample_id=IntSlider(value=1, min=1, max=30, step=1, description="sample #"));

## 5. What makes a training set "bad"?

We want the gap to be small. The slides show this is guaranteed **if** the training set is
a *faithful proxy* of the universe for **every** hypothesis at once:

$$\mathcal{D}_{train}\ \text{is good} \;\iff\; \forall h\in\mathcal{H}:\; \big|L(h,\mathcal{D}_{train}) - L(h,\mathcal{D}_{all})\big| \le \varepsilon.$$

So a training set is **bad** if *at least one* $h$ has a big train-vs-true mismatch. Why
"at least one"? Because the optimizer is free to *pick* that misleading $h$ — one bad
hypothesis is enough to fool us. We now estimate, by brute-force simulation, how often a
random draw is bad, and compare it to the theory.

**Hoeffding's inequality** bounds the failure for a *single* $h$: $P(\text{bad due to }h)\le 2e^{-2N\varepsilon^2}$.
The **union bound** sums over all $|\mathcal{H}|$ hypotheses:

$$P(\mathcal{D}_{train}\ \text{is bad}) \;\le\; \sum_{h\in\mathcal{H}} P(\text{bad due to } h) \;\le\; |\mathcal{H}|\cdot 2e^{-2N\varepsilon^2}.$$

In [ ]:
# === Monte-Carlo P(bad) vs. the Hoeffding union bound ===================================
def hoeffding_bound(H_size, N, eps):
    """Union bound + Hoeffding:  P(D_train is bad) <= |H| * 2 * exp(-2 N eps^2)."""
    return H_size * 2.0 * np.exp(-2.0 * N * eps**2)

def estimate_p_bad(H_size, N, eps, trials=150, seed=0):
    """Fraction of random training sets that are 'bad' (some h with |L_tr - L_all| > eps)."""
    g = np.random.default_rng(seed)
    cand = np.linspace(E_MIN, E_MAX, H_size).astype(int)   # |H| candidate thresholds
    L_all = threshold_loss_curve(cand, e_all, y_all)       # true loss per candidate
    bad = 0
    for _ in range(trials):
        idx  = g.choice(len(e_all), size=N, replace=False)
        L_tr = threshold_loss_curve(cand, e_all[idx], y_all[idx])
        if np.max(np.abs(L_tr - L_all)) > eps:
            bad += 1
    return bad / trials

def bound_vs_empirical(H_size=100, eps=0.10, trials=150):
    Ns  = np.array([50, 100, 200, 300, 500, 800, 1200])
    Ns  = Ns[Ns <= len(e_all)]
    emp = [estimate_p_bad(H_size, n, eps, trials, seed=n) for n in Ns]
    bnd = [min(1.0, hoeffding_bound(H_size, n, eps)) for n in Ns]  # it's a probability -> clip at 1
    plt.figure()
    plt.plot(Ns, bnd, "o-", label="Hoeffding bound (clipped at 1)")
    plt.plot(Ns, emp, "s-", label="empirical P(bad)  [Monte-Carlo]")
    plt.xlabel("N = training-set size"); plt.ylabel("P(D_train is bad)")
    plt.title(f"|H|={H_size},  eps={eps}:  the bound is loose but ALWAYS holds")
    plt.ylim(-0.05, 1.05); plt.legend(); plt.show()
    print("Takeaway: empirical P(bad) stays under the bound, and both shrink as N grows.")

interact(bound_vs_empirical,
         H_size=IntSlider(value=100, min=2, max=400, step=2, description="|H|"),
         eps=FloatSlider(value=0.10, min=0.02, max=0.30, step=0.01, description="eps"),
         trials=IntSlider(value=150, min=50, max=400, step=50, description="trials"));

## 6. How much data do we need? (sample complexity)

Set a tolerable failure probability $\delta$ and solve $|\mathcal{H}|\cdot 2e^{-2N\varepsilon^2}\le\delta$ for $N$:

$$N \;\ge\; \frac{\ln\!\big(2|\mathcal{H}|/\delta\big)}{2\varepsilon^2}.$$

The slide's example $|\mathcal{H}|=10000,\ \delta=0.1,\ \varepsilon=0.1$ gives $N\ge 610$.
The widget reproduces that, and shows $N$ growing only **logarithmically** in $|\mathcal{H}|$ —
encouraging news for complex models.

In [ ]:
# === Required training size to guarantee P(bad) <= delta ================================
def n_required(H_size, delta, eps):
    """Smallest N with the guarantee:  N >= ln(2|H|/delta) / (2 eps^2)."""
    return np.log(2 * H_size / delta) / (2 * eps**2)

def sample_complexity(H_size=10000, delta=0.10, eps=0.10):
    H_size = int(round(H_size))          # FloatLogSlider hands us a float; |H| is a count
    N = n_required(H_size, delta, eps)
    print(f"|H| = {H_size},  delta = {delta},  eps = {eps}")
    print(f"  => need at least N >= {np.ceil(N):.0f} training examples")
    print(f"  (slide check: |H|=10000, delta=0.1, eps=0.1  ->  N >= 610)")

    Hs = np.logspace(1, 6, 60)
    plt.figure()
    plt.plot(Hs, n_required(Hs, delta, eps), lw=2)
    plt.scatter([H_size], [N], color="red", zorder=5, label=f"your setting: N≈{np.ceil(N):.0f}")
    plt.xscale("log")
    plt.xlabel("|H|  (model complexity, log scale)"); plt.ylabel("required N")
    plt.title("Bigger hypothesis class needs more data — but only logarithmically")
    plt.legend(); plt.show()

interact(sample_complexity,
         H_size=FloatLogSlider(value=10000, base=10, min=1, max=6, step=1, description="|H|"),
         delta=FloatSlider(value=0.10, min=0.01, max=0.50, step=0.01, description="delta"),
         eps=FloatSlider(value=0.10, min=0.02, max=0.30, step=0.01, description="eps"));

## 7. The punchline: the model-complexity trade-off

"Why not just use a tiny $|\mathcal{H}|$ so the bound is always safe?" Because a class that is
too small can't fit the data well in the first place — $L(h_{all}, \mathcal{D}_{all})$ goes **up**.

To see both effects with **one knob**, we swap the single threshold for a *binning classifier*:
chop the edge axis into $M$ bins and let each bin vote for its majority training label.
With $M$ bins, $|\mathcal{H}| = 2^M$, so $M$ is a direct dial on model complexity.

- **$M$ too small (underfit):** can't capture the boundary → both losses high.
- **$M$ too large (overfit):** bins get so fine they memorize training noise → training loss
  keeps dropping but **true loss turns back up**.

That U-shaped true-loss curve is the slides' 魚與熊掌 ("fish vs. bear's paw") dilemma — and
the reason the slides end with *"Yes, Deep Learning."*

In [ ]:
# === Bias-complexity trade-off via a binning classifier (|H| = 2^M) ====================
def bin_classifier_loss(M, e_tr, y_tr, e_eval, y_eval):
    """Split [E_MIN, E_MAX] into M bins; each bin predicts its TRAINING-majority label."""
    edges = np.linspace(E_MIN, E_MAX, M + 1)
    b_tr  = np.clip(np.digitize(e_tr, edges) - 1, 0, M - 1)
    ones  = np.bincount(b_tr, weights=y_tr, minlength=M)   # # Digimon per bin (training)
    cnt   = np.bincount(b_tr,               minlength=M)   # # examples per bin (training)
    bin_label = (ones > (cnt - ones)).astype(int)          # majority vote; ties -> Pokémon(0)
    b_ev  = np.clip(np.digitize(e_eval, edges) - 1, 0, M - 1)
    return float(np.mean(bin_label[b_ev] != y_eval))

def complexity_tradeoff(N=200, sample_id=1):
    e_tr, y_tr = sample_train(N, seed=sample_id)
    Ms = np.arange(1, 41)
    train_loss = [bin_classifier_loss(M, e_tr, y_tr, e_tr,  y_tr)  for M in Ms]
    true_loss  = [bin_classifier_loss(M, e_tr, y_tr, e_all, y_all) for M in Ms]

    best_M = Ms[int(np.argmin(true_loss))]
    plt.figure()
    plt.plot(Ms, train_loss, "o-", label="training loss  L(h_train, D_train)")
    plt.plot(Ms, true_loss,  "s-", label="true loss  L(h_train, D_all)")
    plt.axhline(L_h_all, color="grey", ls=":", label=f"best threshold ≈ {L_h_all:.2f}")
    plt.axvline(best_M, color="purple", ls="--", label=f"sweet spot M≈{best_M}")
    plt.xlabel("M = number of bins   (|H| = 2^M,  more complex →)")
    plt.ylabel("loss = error rate")
    plt.title(f"Trade-off (N={N}): training loss always drops, true loss is U-shaped")
    plt.legend(); plt.show()
    print("Underfit (small M): both losses high.   Overfit (large M): train→0 but true loss rises.")

interact(complexity_tradeoff,
         N=IntSlider(value=200, min=30, max=2000, step=10, description="N"),
         sample_id=IntSlider(value=1, min=1, max=20, step=1, description="sample #"));

## 8. 🧠 Questions for you to try

Use the widgets above to investigate — predict first, then check.

1. **Gap hunting.** In §4, fix `N=50` and scrub `sample #` from 1→30. How wildly does the
   true loss swing? Now set `N=1500`. What happened to the swings, and which term of the
   bound explains it?
2. **Bound vs. reality.** In §5, is the empirical `P(bad)` *ever* above the Hoeffding line?
   Why is the bound so loose — what does the union bound assume about the hypotheses that
   is rarely true in practice?
3. **The danger of choice.** Re-derive in words why we union-bound over *all* $h$ rather
   than just the one we end up choosing. (Hint: the optimizer sees $\mathcal{D}_{train}$, not $\mathcal{D}_{all}$.)
4. **Data budget.** In §6, you need ~610 examples for $|\mathcal{H}|=10^4$. If your model
   suddenly has $|\mathcal{H}|=10^{12}$, roughly how many more examples do you need — 10×? 2×?
   Read it off the curve and reconcile with the $\ln|\mathcal{H}|$ term.
5. **Sweet spot.** In §7, how does the best $M$ move as you raise `N` from 50 to 2000? Argue
   from the bound why *more data lets you safely afford a more complex model*.
6. **Break it.** Edit `make_universe` so the two Gaussians barely overlap (push the means
   apart). What happens to $L(h_{all}, \mathcal{D}_{all})$ and to the whole trade-off curve?

## 9. 🚀 Suggested extensions

- **Cross-entropy loss.** The theory only needs the loss to live in $[0,1]$. Swap the 0/1
  error for a clipped cross-entropy and re-check whether the bound still holds empirically.
- **Tighter bounds.** The union bound is wasteful when hypotheses are similar. Read about
  the **VC dimension** and **Rademacher complexity**, which replace $|\mathcal{H}|$ with an
  *effective* complexity for continuous parameter spaces.
- **Two thresholds.** Generalize the classifier to an *interval* rule (Digimon iff $a\le e\le b$),
  making $|\mathcal{H}|\approx|\mathcal{H}_{single}|^2$. Predict the effect on the bound, then verify.
- **Real images.** Replace the synthetic `e` with an actual edge-density feature
  (`cv2.Canny` count) computed on the Kaggle Pokémon and Qiita Digimon image sets the
  slides cite, and see whether the 0.28 floor reappears.
- **Why deep learning wins.** Investigate how over-parameterized networks can have huge
  $|\mathcal{H}|$ yet still generalize — the open question the slides tease with *"Yes, Deep Learning."*